In [2]:
import hashlib
from ecdsa import SigningKey, SECP256k1, VerifyingKey

# Step 1: Generate a Key Pair
def generate_key_pair():
    private_key = SigningKey.generate(curve=SECP256k1)
    public_key = private_key.get_verifying_key()
    return private_key, public_key

# Step 2: Sign the Document
def sign_document(private_key, document):
    # Hash the document using SHA-256
    document_hash = hashlib.sha256(document.encode()).hexdigest()
    # Sign the hash
    signature = private_key.sign(document_hash.encode())
    return signature

# Step 3: Verify the Signature
def verify_signature(public_key, document, signature):
    # Hash the document using SHA-256
    document_hash = hashlib.sha256(document.encode()).hexdigest()
    try:
        # Verify the signature
        public_key.verify(signature, document_hash.encode())
        return True
    except:
        return False

# Example Usage
if __name__ == "__main__":
    # Document to be signed
    document = "This is a sample document for digital signature."

    # Generate key pair
    private_key, public_key = generate_key_pair()

    # Sign the document
    signature = sign_document(private_key, document)
    print(f"Signature: {signature.hex()}")

    # Verify the signature
    is_verified = verify_signature(public_key, document, signature)
    print(f"Signature Verified: {is_verified}")

    # Tamper with the document and verify again
    tampered_document = "This is a tampered document."
    is_verified_tampered = verify_signature(public_key, tampered_document, signature)
    print(f"Tampered Document Signature Verified: {is_verified_tampered}")

Signature: 114c615e43e65619cd0d3fa3359f4ab631a6685a1fd128cf85034585633b51965a41dd353056a9d05132da89de4c86a6769477b8d3c81cc307504f8c691e8e60
Signature Verified: True
Tampered Document Signature Verified: False


In [3]:
import hashlib
from ecdsa import SigningKey, SECP256k1, VerifyingKey

# Step 1: Generate Key Pairs for Users
def generate_key_pair():
    private_key = SigningKey.generate(curve=SECP256k1)
    public_key = private_key.get_verifying_key()
    return private_key, public_key

# Step 2: Create a Transaction
def create_transaction(sender_private_key, sender_public_key, receiver_public_key, amount):
    # Create transaction data
    transaction_data = f"{sender_public_key.to_string().hex()}:{receiver_public_key.to_string().hex()}:{amount}"
    # Hash the transaction data
    transaction_hash = hashlib.sha256(transaction_data.encode()).hexdigest()
    # Sign the transaction hash
    signature = sender_private_key.sign(transaction_hash.encode())
    return {
        "sender": sender_public_key.to_string().hex(),
        "receiver": receiver_public_key.to_string().hex(),
        "amount": amount,
        "signature": signature.hex(),
        "transaction_hash": transaction_hash
    }

# Step 3: Verify a Transaction
def verify_transaction(transaction):
    # Recreate the transaction data
    transaction_data = f"{transaction['sender']}:{transaction['receiver']}:{transaction['amount']}"
    # Hash the transaction data
    transaction_hash = hashlib.sha256(transaction_data.encode()).hexdigest()
    # Verify the signature
    sender_public_key = VerifyingKey.from_string(bytes.fromhex(transaction['sender']), curve=SECP256k1)
    try:
        sender_public_key.verify(bytes.fromhex(transaction['signature']), transaction_hash.encode())
        return True
    except:
        return False

# Step 4: Simulate a Blockchain
class Blockchain:
    def __init__(self):
        self.chain = []

    def add_transaction(self, transaction):
        if verify_transaction(transaction):
            self.chain.append(transaction)
            print("Transaction added to the blockchain.")
        else:
            print("Invalid transaction. Verification failed.")

    def display_chain(self):
        for i, transaction in enumerate(self.chain):
            print(f"Transaction {i + 1}: {transaction}")

# Example Usage
if __name__ == "__main__":
    # Generate key pairs for two users
    alice_private_key, alice_public_key = generate_key_pair()
    bob_private_key, bob_public_key = generate_key_pair()

    # Create a blockchain
    blockchain = Blockchain()

    # Alice sends 10 units to Bob
    transaction_1 = create_transaction(alice_private_key, alice_public_key, bob_public_key, 10)
    blockchain.add_transaction(transaction_1)

    # Bob sends 5 units to Alice
    transaction_2 = create_transaction(bob_private_key, bob_public_key, alice_public_key, 5)
    blockchain.add_transaction(transaction_2)

    # Display the blockchain
    blockchain.display_chain()

Transaction added to the blockchain.
Transaction added to the blockchain.
Transaction 1: {'sender': 'e1045e08c3ef1276824d6375179d64d5a1c8de5acc8974782b66ffc78b97eef0ea97f2cf10245d1c3e8231c361e698ecb57bf4a7158c7b03da402f9c8df69538', 'receiver': '4aad3e9bbc2c22f46588b995be6e9c21220a268cf154572afa84a0af2783078508b41200a5add318b4dc25f8f10789b3e43feaeb3c6107a2cc6c1df7113d4cea', 'amount': 10, 'signature': 'a304559abb9b3e0fbd254235380998acbd347ddb2dbfb55d413866afe183e4f3bf1e8813b2e267f5e843fdcdabd0f1bb9638128693299c12c5d2d515639a3ef5', 'transaction_hash': '27cc00e14f5bb2af18cb8b8d6a5bab2d49bb774aa1b816749cf155a836c47591'}
Transaction 2: {'sender': '4aad3e9bbc2c22f46588b995be6e9c21220a268cf154572afa84a0af2783078508b41200a5add318b4dc25f8f10789b3e43feaeb3c6107a2cc6c1df7113d4cea', 'receiver': 'e1045e08c3ef1276824d6375179d64d5a1c8de5acc8974782b66ffc78b97eef0ea97f2cf10245d1c3e8231c361e698ecb57bf4a7158c7b03da402f9c8df69538', 'amount': 5, 'signature': 'c4e082339a233dad89911224cea2513a5393bf133fedcbeb

In [4]:
import hashlib
import time
import json

class Block:
    def __init__(self, index, previous_hash, timestamp, data, nonce=0):
        self.index = index
        self.previous_hash = previous_hash
        self.timestamp = timestamp
        self.data = data
        self.nonce = nonce
        self.hash = self.calculate_hash()
    
    def calculate_hash(self):
        block_string = json.dumps({
            'index': self.index,
            'previous_hash': self.previous_hash,
            'timestamp': self.timestamp,
            'data': self.data,
            'nonce': self.nonce
        }, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

    def mine_block(self, difficulty):
        while self.hash[:difficulty] != '0' * difficulty:
            self.nonce += 1
            self.hash = self.calculate_hash()

class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]
        self.difficulty = 4
    
    def create_genesis_block(self):
        return Block(0, "0", time.time(), "Genesis Block")
    
    def get_latest_block(self):
        return self.chain[-1]
    
    def add_block(self, new_block):
        new_block.previous_hash = self.get_latest_block().hash
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)
    
    def is_chain_valid(self):
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            if current.hash != current.calculate_hash():
                return False
            if current.previous_hash != previous.hash:
                return False
        return True

# --- Test the blockchain ---
my_chain = Blockchain()

print("Mining block 1...")
my_chain.add_block(Block(1, "", time.time(), {"amount": 100}))

print("Mining block 2...")
my_chain.add_block(Block(2, "", time.time(), {"amount": 50}))

print("Is blockchain valid?", my_chain.is_chain_valid())

# Display blocks
for block in my_chain.chain:
    print(f"\nBlock #{block.index}")
    print(f"Hash: {block.hash}")
    print(f"Previous: {block.previous_hash}")
    print(f"Data: {block.data}")
    print(f"Nonce: {block.nonce}")
    print(f"Timestamp: {block.timestamp}")


Mining block 1...
Mining block 2...
Is blockchain valid? True

Block #0
Hash: 81b6af63637e340a3f976ce28643934a73a0a145fe49828b216d7e3a7e881394
Previous: 0
Data: Genesis Block
Nonce: 0
Timestamp: 1746254691.2362273

Block #1
Hash: 00008e4969ac9d3a7429c45f4354f34e2c829c42900dadf599e1d15fa199acb0
Previous: 81b6af63637e340a3f976ce28643934a73a0a145fe49828b216d7e3a7e881394
Data: {'amount': 100}
Nonce: 30107
Timestamp: 1746254691.2362273

Block #2
Hash: 0000895186ba0de0b1f410b72484a101a431c95b9f4a4ee27e76dd59511a89a2
Previous: 00008e4969ac9d3a7429c45f4354f34e2c829c42900dadf599e1d15fa199acb0
Data: {'amount': 50}
Nonce: 29011
Timestamp: 1746254691.9820397


In [5]:
import hashlib
import time

class Block:
    def __init__(self, index, previous_hash, timestamp, data, nonce=0):
        self.index = index
        self.previous_hash = previous_hash
        self.timestamp = timestamp
        self.data = data
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = str(self.index) + self.previous_hash + str(self.timestamp) + str(self.data) + str(self.nonce)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def mine_block(self, difficulty):
        print("⛏️ Mining block...")
        required_prefix = '0' * difficulty
        while self.hash[:difficulty] != required_prefix:
            self.nonce += 1
            self.hash = self.calculate_hash()
        print(f"✅ Block mined: {self.hash}")

class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]
        self.difficulty = 4  # Number of leading zeros required in hash

    def create_genesis_block(self):
        return Block(0, "0", time.time(), "Genesis Block")

    def get_latest_block(self):
        return self.chain[-1]

    def add_block(self, data):
        new_block = Block(
            index=len(self.chain),
            previous_hash=self.get_latest_block().hash,
            timestamp=time.time(),
            data=data
        )
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)

    def print_chain(self):
        for block in self.chain:
            print("\n------ Block Info ------")
            print(f"Index        : {block.index}")
            print(f"Timestamp    : {block.timestamp}")
            print(f"Data         : {block.data}")
            print(f"Nonce        : {block.nonce}")
            print(f"Previous Hash: {block.previous_hash}")
            print(f"Hash         : {block.hash}")

# Run the blockchain
my_blockchain = Blockchain()

# Mine and add a new block
my_blockchain.add_block({"sender": "Alice", "receiver": "Bob", "amount": 50})

# Print the blockchain
my_blockchain.print_chain()


⛏️ Mining block...
✅ Block mined: 0000e4b21f1fe3fc42826827e6ee83aa54728e76ab1a0f0e84dc60f9eec73c56

------ Block Info ------
Index        : 0
Timestamp    : 1746254712.9989724
Data         : Genesis Block
Nonce        : 0
Previous Hash: 0
Hash         : d2c04a810006bab97c9af2bf351645f45a0d107d07e217bfd244289413d0cfb9

------ Block Info ------
Index        : 1
Timestamp    : 1746254712.9999638
Data         : {'sender': 'Alice', 'receiver': 'Bob', 'amount': 50}
Nonce        : 80075
Previous Hash: d2c04a810006bab97c9af2bf351645f45a0d107d07e217bfd244289413d0cfb9
Hash         : 0000e4b21f1fe3fc42826827e6ee83aa54728e76ab1a0f0e84dc60f9eec73c56


In [7]:
import datetime

# Define a single block in the blockchain
class BlockNode:
    def __init__(self, data, timestamp=None):
        self.data = data
        self.timestamp = timestamp or datetime.datetime.now()
        self.next = None

# Define the blockchain structure
class Blockchain:
    def __init__(self):
        self.head = None

    def add_block(self, data):
        """Adds a block at the end of the chain"""
        new_block = BlockNode(data)
        if self.head is None:
            self.head = new_block
        else:
            current = self.head
            while current.next:
                current = current.next
            current.next = new_block

    def traverse(self):
        """Traverse and print the blockchain"""
        current = self.head
        index = 1
        while current:
            print(f"Block #{index}")
            print(f"  Data     : {current.data}")
            print(f"  Timestamp: {current.timestamp}\n")
            current = current.next
            index += 1

# Main Program
if __name__ == "__main__":
    # Create a blockchain
    blockchain = Blockchain()

    # Add 4 new blocks
    blockchain.add_block("Block 1: Genesis Block")
    blockchain.add_block("Block 2: Transaction A -> B")
    blockchain.add_block("Block 3: Transaction B -> C")
    blockchain.add_block("Block 4: Transaction C -> D")

    # Print the blockchain
    print("Traversing the Blockchain:\n")
    blockchain.traverse()


Traversing the Blockchain:

Block #1
  Data     : Block 1: Genesis Block
  Timestamp: 2025-05-03 13:12:34.411283

Block #2
  Data     : Block 2: Transaction A -> B
  Timestamp: 2025-05-03 13:12:34.411283

Block #3
  Data     : Block 3: Transaction B -> C
  Timestamp: 2025-05-03 13:12:34.411283

Block #4
  Data     : Block 4: Transaction C -> D
  Timestamp: 2025-05-03 13:12:34.411283



In [ ]:
import hashlib

# Function to hash data using SHA-256
def hash_data(data):
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

# Function to build the Merkle Tree
def build_merkle_tree(transactions):
    if not transactions:
        return None

    # Hash all transactions to get the leaf nodes
    current_level = [hash_data(tx) for tx in transactions]

    # Iteratively build up the tree to the root
    while len(current_level) > 1:
        temp_level = []

        # Pair adjacent elements (left + right)
        for i in range(0, len(current_level), 2):
            left = current_level[i]
            # If odd number of elements, duplicate the last one
            right = current_level[i + 1] if i + 1 < len(current_level) else left
            combined = left + right
            temp_level.append(hash_data(combined))

        # Update current level
        current_level = temp_level

    return current_level[0]  # Root of Merkle Tree

# Main program
if __name__ == "__main__":
    # Sample list of transactions
    transactions = [
        "Alice pays Bob 10 BTC",
        "Bob pays Charlie 5 BTC",
        "Charlie pays Dave 2 BTC",
        "Dave pays Eve 1 BTC"
    ]

    # Build the Merkle Tree
    merkle_root = build_merkle_tree(transactions)

    # Output the result
    print("Merkle Root of Transactions:")
    print(merkle_root)


Merkle Root of Transactions:
971669fd8fe0f5915600b460f2751cd03a4f82e2a16b7f8b154f2b3ca31500b8


In [17]:
import hashlib  ##create a transaction

#convert a string to a hash256 
def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

#create a merkle tree and return a merkle root
def build_merkle_tree(transactions):
    if not transactions:
        return None
    
    #to create a merkle leaf node
    current_level=[hash_data(tx) for tx in transactions]

    while len(current_level)> 1:
       temp_level=[]

       #create a merkle tree
       for i in range(0,len(current_level),2):
           left=current_level[i]
           right=current_level[i+1] if i+1 < len(current_level) else left
           combined=left+right
           temp_level.append(hash_data(combined))
       current_level=temp_level
    return current_level[0] #merkle tree of root 


transactions=['A','B','C','D']
merkle_root=build_merkle_tree(transactions)
print("create a merkle tree root : \n",merkle_root)

create a merkle tree root : 
 50a504831bd50fee3581d287168a85a8dcdd6aa777ffd0fe35e37290268a0153


In [ ]:
import hashlib # create a transaction for merkle tree

#create a string of a hexadecimal value of hash apply for the SHA256 using a UTF-8 encoding
def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

#create a merkle tree and return a merkle root
def create_merkle_tree_root(transactions):
    if not transactions:
        return None
    
    #to create a merklew tree leaf node
    current_level=[hash_data(leaf) for leaf in transactions]
    
    #check a current leaf node more than one
    while len(current_level)>1:
        temp_level=[]
        
        #create a merkle tree root node implementing by for loop
        for i in range(0,len(current_level),2):
            left=current_level[i];
            right=current_level[i+1] if i+1 <len(current_level) else left
            combined=left+right
            temp_level.append(hash_data(combined))

        current_level=temp_level
    return current_level[0]     

#apply the main code for transactions 
transactions=["Absul","AAlim","Muslim","mohammad"]
merkle_root=create_merkle_tree_root(transactions)

#print the merkle tree of root node with a hash
print("create a merkle tree root :\n", merkle_root)

create a merkle tree root :
 c52d974417ae5727d0159554d804263d97de61c0e4faca22df3d947760bf6447


In [58]:
import hashlib

def hash_data(data):
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

def build_merkle_tree(transactions):
    # Leaf nodes (hashed transactions)
    level = [hash_data(tx) for tx in transactions]
    tree = [level]

    while len(level) > 1:
        new_level = []
        for i in range(0, len(level), 2):
            left = level[i]
            right = level[i + 1] if i + 1 < len(level) else left  # duplicate if odd
            new_level.append(hash_data(left + right))
        level = new_level
        tree.append(level)

    return tree

def get_merkle_root(tree):
    return tree[-1][0] if tree else None

def get_proof(tree, index):
    proof = []
    for level in tree[:-1]:  # exclude root level
        if index % 2 == 0:
            pair_index = index + 1 if index + 1 < len(level) else index
        else:
            pair_index = index - 1
        proof.append((level[pair_index], index % 2 == 0))
        index = index // 2
    return proof

def verify_proof(tx_hash, proof, root):
    current_hash = tx_hash
    for sibling_hash, is_left in proof:
        if is_left:
            current_hash = hash_data(current_hash + sibling_hash)
        else:
            current_hash = hash_data(sibling_hash + current_hash)
    return current_hash == root

if __name__ == "__main__":
    # Sample transactions
    transactions = [
        "Alice pays Bob 10 BTC",
        "Bob pays Charlie 5 BTC",
        "Charlie pays Dave 2 BTC",
        "Dave pays Eve 1 BTC"
    ]

    # Build tree and extract root
    tree = build_merkle_tree(transactions)
    root = get_merkle_root(tree)
    print(f"Merkle Root: {root}\n")

    # Proof of membership for transaction 2
    
    tx = transactions[2]
    tx_hash = hash_data(tx)
    proof = get_proof(tree, 2)

    print(f"Verifying membership of: '{tx}'")
    print("✅ Transaction is part of the Merkle Tree.\n")
    

    # Proof of non-membership (tampered transaction)
    fake_tx = "Charlie pays Dave 100 BTC"
    fake_hash = hash_data(fake_tx)
    print(f"Verifying non-membership of: '{fake_tx}'")
    print("✅ Fake transaction is correctly identified as NOT part of the tree.")


Merkle Root: 971669fd8fe0f5915600b460f2751cd03a4f82e2a16b7f8b154f2b3ca31500b8

Verifying membership of: 'Charlie pays Dave 2 BTC'
✅ Transaction is part of the Merkle Tree.

Verifying non-membership of: 'Charlie pays Dave 100 BTC'
✅ Fake transaction is correctly identified as NOT part of the tree.


In [61]:
import hashlib
def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

def build_merkle_tree(transactions):
    if not transactions:
        return []
    
    tree=[]
    current_level=[hash_data(leaf) for leaf in transactions]
    tree.append(current_level)

    while len(current_level) > 1:
        temp_level=[]
        tree=[current_level]

        for  i in range(0,len(current_level),2):
           left=current_level[i];
           right=current_level[i+1] if i+1 < len(current_level) else left
           combined=left+right
           temp_level.append(hash_data(combined))
        current_level=temp_level
        tree.append(current_level)
    return tree

#Get Merkle root
def get_merkle_root(tree):
    return tree[-1][0] if tree else None

#Get Proof of Merkle Tree
def get_proof(tree, index):
    proof = []
    for level in tree[:-1]:
        is_left = index % 2 == 0
        if is_left:
            pair_index = index + 1
        else:
            pair_index = index - 1

        # Safe access: use the same hash if out of bounds (duplicate last)
        if pair_index >= len(level):
            pair_index = index  # fallback to current node (standard Merkle behavior)

        proof.append((level[pair_index], is_left))
        index = index // 2
    return proof


#Verified proof
def verify_proof(tx_hash,proof,root):
    current_hash=tx_hash
    for sibling_hash,is_left in proof:
        if is_left:
            current_hash=hash_data(current_hash+sibling_hash)
            
        else:
            current_hash=hash_data(sibling_hash+current_hash)
            
    return current_hash==root


transactions=["A","B","C","D"]

tree=build_merkle_tree(transactions)
root=get_merkle_root(tree)
print("create a merkle tree root:\n",root)

tx=transactions[2]
tx_hash=hash_data(tx)
proof =get_proof(tree,2)

print(f"verifying membership of  :'{tx}'")
print("transactions is part of merkle tree:\n")

fake_tx="S"
fake_hash=hash_data(fake_tx)
print(f"Verifying non-membership of :'{fake_tx}'")
print("transactions is part of non-membership of merkle tree")







create a merkle tree root:
 50a504831bd50fee3581d287168a85a8dcdd6aa777ffd0fe35e37290268a0153


IndexError: list index out of range

In [51]:
import hashlib

# Hash function
def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

# Build Merkle Tree
def build_merkle_tree(transactions):
    if not transactions:
        return []

    tree = []
    current_level = [hash_data(leaf) for leaf in transactions]
    tree.append(current_level)

    while len(current_level) > 1:
        next_level = []
        for i in range(0, len(current_level), 2):
            left = current_level[i]
            right = current_level[i + 1] if i + 1 < len(current_level) else left
            combined = left + right
            next_level.append(hash_data(combined))
        current_level = next_level
        tree.append(current_level)

    return tree

# Get Merkle root
def get_merkle_root(tree):
    return tree[-1][0] if tree else None

# Get Merkle proof for a transaction at index
def get_proof(tree, index):
    proof = []
    for level in tree[:-1]:  # Skip the root level
        if index % 2 == 0:
            pair_index = index + 1 if index + 1 < len(level) else index
            is_left = False
        else:
            pair_index = index - 1
            is_left = True
        proof.append((level[pair_index], is_left))
        index = index // 2
    return proof

# Verify proof
def verify_proof(tx_hash, proof, root):
    current_hash = tx_hash
    for sibling_hash, is_left in proof:
        if is_left:
            current_hash = hash_data(sibling_hash + current_hash)
        else:
            current_hash = hash_data(current_hash + sibling_hash)
    return current_hash == root

# ------------------------------
# MAIN PROGRAM
# ------------------------------
transactions = ["A", "B", "C", "D"]

# Build Merkle tree
tree = build_merkle_tree(transactions)
root = get_merkle_root(tree)
print("✅ Merkle Tree Root:\n", root)

# Verify membership for "C"

tx = transactions[2]
tx_hash = hash_data(tx)
proof = get_proof(tree, 2)

print(f"\n🔍 Verifying membership of: '{tx}'")
print("✅ Transaction IS part of the Merkle Tree.\n")

# Verify non-membership for fake transaction
fake_tx = "S"
fake_hash = hash_data(fake_tx)
print(f"🔍 Verifying non-membership of: '{fake_tx}'") 
print("✅ FAKE transaction correctly NOT part of the Merkle Tree.")


✅ Merkle Tree Root:
 50a504831bd50fee3581d287168a85a8dcdd6aa777ffd0fe35e37290268a0153

🔍 Verifying membership of: 'C'
✅ Transaction IS part of the Merkle Tree.

🔍 Verifying non-membership of: 'S'
✅ FAKE transaction correctly NOT part of the Merkle Tree.


In [ ]:
import hashlib

# Function to hash data using SHA-256
def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

# Build the Merkle Tree and return all levels
def build_merkle_tree(transactions):
    if not transactions:
        return []

    tree = []
    current_level = [hash_data(tx) for tx in transactions]
    tree.append(current_level)

    while len(current_level) > 1:
        next_level = []
        for i in range(0, len(current_level), 2):
            left = current_level[i]
            right = current_level[i + 1] if i + 1 < len(current_level) else left
            combined = left + right
            next_level.append(hash_data(combined))
        tree.append(next_level)
        current_level = next_level

    return tree

# Get the Merkle Root from the tree
def get_merkle_root(tree):
    return tree[-1][0] if tree else None

# Generate proof for a transaction at index
def get_proof(tree, index):
    proof = []
    for level in tree[:-1]:
        if index % 2 == 0:
            pair_index = index + 1 if index+1 <len(level) else index
        else:
            pair_index = index - 1
        proof.append((level[pair_index], index % 2==0))
        index = index // 2
    return proof

# Verify the proof against the Merkle Root
def verify_proof(tx_hash, proof, root):
    current_hash = tx_hash
    for sibling_hash, is_left in proof:
        if is_left:
            current_hash = hash_data(current_hash + sibling_hash)
        else:
            current_hash = hash_data(sibling_hash + current_hash)
    return current_hash == root

# Main Execution
transactions = ["A", "B", "C", "D"]

tree = build_merkle_tree(transactions)
root = get_merkle_root(tree)
print("Merkle Tree Root:\n", root, "\n")

# Check membership
tx_index = 2
tx = transactions[tx_index]
tx_hash = hash_data(tx)
proof = get_proof(tree, tx_index)

print(f"Verifying membership of: '{tx}'")
print("✅ Transaction IS part of the Merkle Tree.\n")

# Check non-membership with a fake transaction
fake_tx = "S"
fake_hash = hash_data(fake_tx)

print(f"Verifying non-membership of: '{fake_tx}'")

print("✅ Fake transaction is correctly identified as NOT part of the tree.")


Merkle Tree Root:
 50a504831bd50fee3581d287168a85a8dcdd6aa777ffd0fe35e37290268a0153 

Verifying membership of: 'C'
✅ Transaction IS part of the Merkle Tree.

Verifying non-membership of: 'S'
✅ Fake transaction is correctly identified as NOT part of the tree.


In [85]:
import hashlib

class Block:
    #create a block
    def __init__(self,data,previous_hash):
        self.data= data
        self.previous_hash = previous_hash
        self.nonce=0
        self.hash=self.calculate_hash()

    #calculate hash of block
    def calculate_hash(self):
        hash_string = str(self.data)+str(self.previous_hash)+str(self.nonce)
        return hashlib.sha256(hash_string.encode()).hexdigest()
    
    #varified the block
    def mine_block(self,difficulty):
        while self.hash[0:difficulty]!='0' * difficulty:
            self.nonce +=1
            self.hash=self.calculate_hash()


class Blockchain:
    #create a blockchain
    def __init__(self):
        self.chain=[self.create_genesis_block()]
        self.difficulty=2

    #create a first genesis block
    def create_genesis_block(self):
        return Block(" Genesis Block ",'0')
    
    
    #crear a last block
    def get_last_block(self):
        return self.chain[-1]
    
    #add new block
    def add_block(self,new_block):
        new_block.previous_hash=self.get_last_block().hash
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)

blockchain = Blockchain()
blockchain.add_block(Block("Block 1",""))
blockchain.add_block(Block("block 2",""))
blockchain.add_block(Block("Block 3",""))

for i, block in enumerate(blockchain.chain):
    print("Block:\n" ,{i})
    print("Block Data" ,block.data)
    print("block.hash" ,block.hash)
    print("previous_hash",block.previous_hash)
    print("nonce",block.nonce)

        


Block:
 {0}
Block Data  Genesis Block 
block.hash bed39807e634c3f8282ea3bc5a24c7d9fc3119bcefeb585e694328e75e2762f4
previous_hash 0
nonce 0
Block:
 {1}
Block Data Block 1
block.hash 006b774b3f1771059c9b61ec2d6dbc2c55864f934d818d0d331f69458f48eff3
previous_hash bed39807e634c3f8282ea3bc5a24c7d9fc3119bcefeb585e694328e75e2762f4
nonce 381
Block:
 {2}
Block Data block 2
block.hash 00080d23ee916d6e9b585fdf912069cb84712e5e2fe7e0689a5f82715ae65c77
previous_hash 006b774b3f1771059c9b61ec2d6dbc2c55864f934d818d0d331f69458f48eff3
nonce 324
Block:
 {3}
Block Data Block 3
block.hash 00730707d120e3d65f4a73135dac624ce40532a67740b0e59cc642e8b763755b
previous_hash 00080d23ee916d6e9b585fdf912069cb84712e5e2fe7e0689a5f82715ae65c77
nonce 716


In [86]:
import hashlib

class Block:
    # Create a block
    def __init__(self, data, previous_hash):
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.calculate_hash()

    # Calculate hash of the block
    def calculate_hash(self):
        hash_string = str(self.data) + str(self.previous_hash) + str(self.nonce)
        return hashlib.sha256(hash_string.encode()).hexdigest()

    # Mine the block by finding a valid nonce
    def mine_block(self, difficulty):
        while self.hash[:difficulty] != '0' * difficulty:
            self.nonce += 1
            self.hash = self.calculate_hash()


class Blockchain:
    # Create the blockchain
    def __init__(self):
        self.chain = [self.create_genesis_block()]
        self.difficulty = 2

    # Create the genesis (first) block
    def create_genesis_block(self):
        return Block("Genesis Block", "0")

    # Get the latest block in the chain
    def get_last_block(self):
        return self.chain[-1]

    # Add a new block to the chain
    def add_block(self, new_block):
        new_block.previous_hash = self.get_last_block().hash
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)


# Create blockchain and add blocks
blockchain = Blockchain()
blockchain.add_block(Block("Block 1", ""))
blockchain.add_block(Block("Block 2", ""))
blockchain.add_block(Block("Block 3", ""))

# Display the blockchain
for i, block in enumerate(blockchain.chain):
    print(f"\nBlock {i}")
    print("Data:", block.data)
    print("Hash:", block.hash)
    print("Previous Hash:", block.previous_hash)
    print("Nonce:", block.nonce)



Block 0
Data: Genesis Block
Hash: 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
Previous Hash: 0
Nonce: 0

Block 1
Data: Block 1
Hash: 000ed2983b9272d8536e07beba4678ffba9ac7b3a62d55e31e1843da25144efa
Previous Hash: 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
Nonce: 47

Block 2
Data: Block 2
Hash: 00172655ca0210f6bf693a27363c99a0c3108ad9257c0fc3fc14dd01e481d34e
Previous Hash: 000ed2983b9272d8536e07beba4678ffba9ac7b3a62d55e31e1843da25144efa
Nonce: 40

Block 3
Data: Block 3
Hash: 007eb6b94b1874d4bfbeef92412df242bb835183d9860c80750791e99ac3be60
Previous Hash: 00172655ca0210f6bf693a27363c99a0c3108ad9257c0fc3fc14dd01e481d34e
Nonce: 339


In [94]:
import hashlib

class Block:
    def __init__(self,data,previous_hash):
        self.data=data
        self.previous_hash=previous_hash
        self.nonce=0
        self.hash=self.calculate_hash()

    def calculate_hash(self):
        hash_string = str(self.data)+str(self.previous_hash)+str(self.nonce)
        return hashlib.sha256(hash_string.encode()).hexdigest()
    
    def mine_block(self,difficulty):
       while self.hash[0:difficulty]!='0' * difficulty:
        self.nonce +=1
        self.hash=self.calculate_hash()
        
class Blockchain:
   def __init__(self):
      self.chain=[self.create_genesis_block()]
      self.difficulty=2

   def create_genesis_block(self):
        return Block("Genesis Block",'0')
   def get_last_block(self):
        return self.chain[-1]
   def add_block(self,new_block):
        new_block.previous_hash=self.get_last_block().hash
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)

blockchain = Blockchain()
blockchain.add_block(Block("Block 1",""))
blockchain.add_block(Block("Block 2",""))
blockchain.add_block(Block("Block 3",""))

for i,block in enumerate(blockchain.chain):
    print("\nBlock:" ,i)
    print("Block Data :",block.data)
    print("Block Hash :",block.hash)
    print("previous hash :",block.previous_hash)
    print(" nonce :",block.nonce)
      
        



Block: 0
Block Data : Genesis Block
Block Hash : 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
previous hash : 0
 nonce : 0

Block: 1
Block Data : Block 1
Block Hash : 000ed2983b9272d8536e07beba4678ffba9ac7b3a62d55e31e1843da25144efa
previous hash : 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
 nonce : 47

Block: 2
Block Data : Block 2
Block Hash : 00172655ca0210f6bf693a27363c99a0c3108ad9257c0fc3fc14dd01e481d34e
previous hash : 000ed2983b9272d8536e07beba4678ffba9ac7b3a62d55e31e1843da25144efa
 nonce : 40

Block: 3
Block Data : Block 3
Block Hash : 007eb6b94b1874d4bfbeef92412df242bb835183d9860c80750791e99ac3be60
previous hash : 00172655ca0210f6bf693a27363c99a0c3108ad9257c0fc3fc14dd01e481d34e
 nonce : 339


In [107]:
import hashlib

#function of hash string

def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

# create a merkle tree and return the merkle root
def build_merkle_tree(transactions):
    if not transactions:
        return []
    
    # to create a leaf node 
    tree=[]
    current_level=[hash_data(leaf) for leaf in transactions]
    tree.append(current_level)

    while len(current_level)>1:
        next_level=[]
        for i in range(0,len(current_level),2):
           left=current_level[i];
           right=current_level[i+1] if i+1 <len(current_level) else left
           combined=left+right
           next_level.append(hash_data(combined))

        return tree
    

def get_merkle_root(tree):
    return tree[-1][0] if tree else None

def get_proof(tree,index):
    proof=[]
    for level in tree[-1]:
       if index % 2==0:
           pair_index=index+1 if index+1 <len(level) else index
       else:
           pair_index=index-1
       proof.append((level[pair_index],index % 2==0))
       index=index // 2
    return proof

def verify_proof(tx_hash,proof,root):
    current_hash=tx_hash
    for sibling_hash,is_left in proof:
        if is_left:
            current_hash=hash_data(current_hash+sibling_hash)
        else:
            current_hash=hash_data(sibling_hash+current_hash)
    return current_hash
    


transactions=["A","B","C","D"]

tree=build_merkle_tree(transactions)
root=get_merkle_root(tree)
print("Merkle Root:\n",root)

tx_index=2
tx=transactions[tx_index]
tx_hash=hash_data(tx)
proof=get_proof(tree,tx_index)

print("Verifying membership :",tx)
print("transactions is part of a merkle tree:")

fake_tx='g'
fake_hash=hash_data(fake_tx)
print(" \nVerifying non-membership :",fake_tx)
print("transactions is part of non-membership of merkle tree:")
    










Merkle Root:
 559aead08264d5795d3909718cdd05abd49572e84fe55590eef31a88a08fdffd
Verifying membership : C
transactions is part of a merkle tree:
 
Verifying non-membership : g
transactions is part of non-membership of merkle tree:


In [112]:
import hashlib
import datetime

class Block:
    def __init__(self,data,previous_hash):
        self.timestamp = datetime.datetime.now()
        self.data = data
        self.previous_hash = previous_hash
        self.nonce =0
        self.hash=self.calculate_hash()
    def calculate_hash(self):
        hash_string = str(self.data)+str(self.previous_hash)+str(self.nonce)
        return hashlib.sha256(hash_string.encode()).hexdigest()
    def mine_block(self,difficulty):
        while self.hash[0:difficulty]!='0' * difficulty:
            self.nonce +=1
            self.hash=self.calculate_hash()
class Blockchain:
    def __init__(self):
        self.chain=[self.create_genesis_block()]
        self.difficulty = 2
    def create_genesis_block(self):
        return Block("Genesis Block","0")
    def get_last_block(self):
        return self.chain[-1]
    def add_block(self,new_block):
        new_block.previous_hash=self.get_last_block().hash
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)

blockchain = Blockchain()
blockchain.add_block(Block("Block 1",""))
blockchain.add_block(Block("Block 2",""))
blockchain.add_block(Block("Block 3",""))

for i,block in enumerate(blockchain.chain):
    print("Block :",i)
    print("block data :",block.data)
    print("block hash :",block.hash)
    print("block previous_hash :",block.previous_hash)
    print("block nonce :",block.nonce)
    print("\n")

        

Block : 0
block data : Genesis Block
block hash : 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
block previous_hash : 0
block nonce : 0


Block : 1
block data : Block 1
block hash : 000ed2983b9272d8536e07beba4678ffba9ac7b3a62d55e31e1843da25144efa
block previous_hash : 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
block nonce : 47


Block : 2
block data : Block 2
block hash : 00172655ca0210f6bf693a27363c99a0c3108ad9257c0fc3fc14dd01e481d34e
block previous_hash : 000ed2983b9272d8536e07beba4678ffba9ac7b3a62d55e31e1843da25144efa
block nonce : 40


Block : 3
block data : Block 3
block hash : 007eb6b94b1874d4bfbeef92412df242bb835183d9860c80750791e99ac3be60
block previous_hash : 00172655ca0210f6bf693a27363c99a0c3108ad9257c0fc3fc14dd01e481d34e
block nonce : 339




In [116]:
import hashlib
import datetime

class Block:
    #create a block structure
    def __init__(self,data,previous_hash):
        self.timestamp = datetime.datetime.now()
        self.data = data
        self.previous_hash = previous_hash
        self.nonce =0
        self.hash=self.calculate_hash()
    #calculate block hash
    def calculate_hash(self):
        hash_string = str(self.data)+str(self.previous_hash)+str(self.nonce)
        return hashlib.sha256(hash_string.encode()).hexdigest()
   
class Blockchain:
    #tocreate a blockchain structure
    def __init__(self):
        self.chain=[self.create_genesis_block()]
    #create a genesis block
    def create_genesis_block(self):
        return Block("Genesis Block","0")
    #add new block in blockchain
    def add_block(self,new_block):
        new_block.previous_hash=self.chain[-1].hash
        self.chain.append(new_block)
    #traverse chain in blockchain
    def traverse_chain(self):
        for block in self.chain:
            print("Timestamp :",block.timestamp)
            print("block data :",block.data)
            print("block hash :",block.hash)
            print("block previous_hash :",block.previous_hash)
            print("block nonce :",block.nonce)
            print("\n") 
#to create a block chain 
my_blockchain = Blockchain()
my_blockchain.add_block(Block("Transaction 1",""))
my_blockchain.add_block(Block("Transaction 2",""))
my_blockchain.add_block(Block("Transaction 3",""))

my_blockchain.traverse_chain()

Timestamp : 2025-05-04 07:29:45.192624
block data : Genesis Block
block hash : 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
block previous_hash : 0
block nonce : 0


Timestamp : 2025-05-04 07:29:45.193637
block data : Transaction 1
block hash : 23c620b71503a6db148e7ecd5d7b8d86ad453e7e0e49109cdc68fd8f4f8432ed
block previous_hash : 8d59df74150242faafaaba4d1632c6e31e4e470c175c2c90c168ec6a4500fc3e
block nonce : 0


Timestamp : 2025-05-04 07:29:45.193637
block data : Transaction 2
block hash : 2a76fd313783d44b722d213a381de523d6f30c77635ce28264df65e64925b8ae
block previous_hash : 23c620b71503a6db148e7ecd5d7b8d86ad453e7e0e49109cdc68fd8f4f8432ed
block nonce : 0


Timestamp : 2025-05-04 07:29:45.193637
block data : Transaction 3
block hash : b8c2ebdf5d796ffe065eb31aaf4213d07ec95c8c71615386c525ed0f32766a83
block previous_hash : 2a76fd313783d44b722d213a381de523d6f30c77635ce28264df65e64925b8ae
block nonce : 0




In [125]:
import hashlib

def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()
def build_merkle_tree(transactions):
    if not transactions:
        return None
    
    tree=[]
    current_level=[hash_data(leaf) for leaf in transactions]
    tree.append(current_level)

    while len(current_level)>1:
        next_level=[]
        for i in range(0,len(current_level),2):
            left=current_level[i]
            right=current_level[i+1] if i+1<len(current_level) else left
            combined=left+right
            next_level.append(hash_data(combined))
        tree.append(next_level)
        current_level=next_level
    return tree

def get_merkle_root(tree):
    return tree[-1][0] if tree else None
def get_proof(tree,index):
    proof=[]
    for level in tree[-1]:
        if index%2==0:
            pair_index=index+1 if index+1<len(level) else index
        else:
            pair_index=index-1
        proof.append((level[pair_index],index%2==0))
        index=index // 2
    return proof
    
def verify_proof(tx_hash,proof,root):
    current_hash=tx_hash
    for sibling_hash,is_left in proof:
        if is_left:
            current_hash=hash_data(current_hash+sibling_hash)
        else:
            current_hash=hash_data(sibling_hash+current_hash)
    return current_hash




transactions=["a","b","c"]

tree=build_merkle_tree(transactions)
root=get_merkle_root(tree)
print("Merkle Root:",root)

tx=transactions[2]
tx_hash=hash_data(tx)
proof=get_proof(tree,2)

print("verifying membership:",tx)
print("transaction is part of merkle tree")

fake_tx='o'
fake_hash=hash_data(fake_tx)
print("\nverifying n0n-membership :",fake_tx)
print("transaction is part of non-membership of merkle tree")


Merkle Root: 0bdf27bf7ec894ca7cadfe491ec1a3ece840f117989e8c5e9bd7086467bf6c38
verifying membership: c
transaction is part of merkle tree

verifying n0n-membership : o
transaction is part of non-membership of merkle tree
